In [1]:
from transformers import GPT2LMHeadModel

In [2]:
model_hf = GPT2LMHeadModel.from_pretrained('gpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
state_hf = model_hf.state_dict()
for k, v in state_hf.items():
  print(k, v.shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

In [4]:
import torch.nn as nn
import torch

In [5]:
device = "cpu"
if torch.cuda.is_available():
  device = "cuda"
print('Device is ' + str(device))

Device is cuda


In [6]:
from torch import embedding
from dataclasses import dataclass

@dataclass
class GPTConfig:
  num_heads = 6
  num_layers = 6
  vocab_size = 50257
  embedding_dim = 768
  block_size = 256
  seq_len = 256
  lag_behind = 1

In [7]:
import torch.nn.functional as F

In [8]:
class LagBehindGPT2(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(
        dict(
            wte = nn.Embedding(config.vocab_size, config.embedding_dim),
            wpe = nn.Embedding(config.block_size, config.embedding_dim),  # one set
            h = nn.ModuleList([LagLayerBlock(config) for _ in range(config.num_layers)]),
            ln_f = nn.LayerNorm(config.embedding_dim),
        )
    )
    self.lm_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

  def forward(self, x, y=None, lam=0.5):
    B, SL = x.size()
    V = self.config.vocab_size
    skip_dist = self.config.lag_behind + 1

    pos = torch.arange(0, SL, dtype=torch.long).to(device)
    pos_emb = self.transformer.wpe(pos)
    tok_emb = self.transformer.wte(x)

    x_shared = pos_emb + tok_emb

    x_pre = x_shared
    x_fut = x_shared

    for layer_block in self.transformer.h:
      x_pre = layer_block(x_pre, False)
      x_fut = layer_block(x_fut, True)

    x_pre = self.transformer.ln_f(x_pre)
    x_fut = self.transformer.ln_f(x_fut)

    logits_pre = self.lm_head(x_pre)
    logits_fut = self.lm_head(x_fut)

    loss = None
    if y is not None:
        # Forward loss: standard next-token prediction
        loss_fwd = F.cross_entropy(logits_pre.view(-1, V), y.view(-1))

        # Backward loss: position i predicts input token at i - skip_dist
        # Skip first skip_dist positions (no valid target exists)
        bwd_targets = x[:, :-skip_dist]        # tokens at indices 0..SL-skip_dist-1
        bwd_logits = logits_fut[:, skip_dist:]  # predictions from positions skip_dist..SL-1
        loss_bwd = F.cross_entropy(bwd_logits.reshape(-1, V), bwd_targets.reshape(-1))

        loss = loss_fwd + lam * loss_bwd

    return logits_pre, logits_fut, loss


# RENAMED HELPER CLASSES BELOW

class LagLayerBlock(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.embedding_dim)
    self.attn = LagAttentionMultiHeadFused(config)
    self.ln_2 = nn.LayerNorm(config.embedding_dim)
    self.mlp = LagMLP(config)

  def forward(self, x, future):
    x = x + self.attn(self.ln_1(x), future)
    x = x + self.mlp(self.ln_2(x))
    return x

class LagMLP(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.c_fc = nn.Linear(config.embedding_dim, 4*config.embedding_dim, bias=False)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4*config.embedding_dim, config.embedding_dim, bias=False)

  def forward(self, x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x

class LagAttentionMultiHeadFused(nn.Module):
  def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.w_qkv = nn.Linear(config.embedding_dim, 3 * config.embedding_dim)
        self.output = nn.Linear(config.embedding_dim, config.embedding_dim)

  def forward(self, x, skip_mask=False):
      B, SL, ED = x.size()
      qkv = self.w_qkv(x)
      q, k, v = qkv.split(self.config.embedding_dim, dim=2)
      q = q.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1, 2)
      k = k.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1, 2)
      v = v.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1, 2)

      if skip_mask:
          skip_dist = self.config.lag_behind + 1
          mask = torch.tril(torch.ones(SL, SL, device=x.device)).bool()
          # Punch out t_{i - skip_dist} for each row i >= skip_dist
          rows = torch.arange(skip_dist, SL, device=x.device)
          mask[rows, rows - skip_dist] = False
          mask = torch.where(mask, 0.0, float('-inf'))
          attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
      else:
          attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

      attn_out = attn_out.transpose(1, 2).contiguous().view(B, SL, ED)
      y = self.output(attn_out)
      return y

In [9]:
class GPT2(nn.Module):
  #create a ModuleDict with wte, wpe, hidden layers, weight and bias
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(
        dict(
            wte = nn.Embedding(config.vocab_size, config.embedding_dim), # how do we determine the dimensions of this?
            wpe = nn.Embedding(config.block_size, config.embedding_dim),
            h = nn.ModuleList([LayerBlock(config) for _ in range(config.num_layers)]),
            ln_f = nn.LayerNorm(config.embedding_dim)
        )
    )
    self.lm_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False) #why is bias set as False?

  def forward(self, x, y=None):
    B, SL = x.size()

    pos = torch.arange(0, SL, dtype=torch.long).to(device) #(SL, )
    pos_emb = self.transformer.wpe(pos) #(SL, ED)
    tok_emb = self.transformer.wte(x) #(B, SL, ED)

    x = pos_emb + tok_emb #(B, SL, ED)

    for layer_block in self.transformer.h:
      x = layer_block(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x) #(B, SL, Vocab)

    loss = None
    if y is not None:
      loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)) #(B*SL, Vocab) | (B*SL, )
    return logits, loss

class LayerBlock(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.embedding_dim)
    self.attn = AttentionMultiHeadFused(config)
    self.ln_2 = nn.LayerNorm(config.embedding_dim)
    self.mlp = MLP(config)

  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x)) #why do we layer-norm before passing mlp?
    return x


class MLP(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.c_fc = nn.Linear(config.embedding_dim, 4*config.embedding_dim, bias=False)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4*config.embedding_dim, config.embedding_dim, bias=False)

  def forward(self, x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x

class AttentionMultiHeadFused(nn.Module):
  def __init__(self, config: GPTConfig):
    super().__init__()
    self.config = config
    self.w_qkv = nn.Linear(config.embedding_dim, 3*config.embedding_dim) #does bias matter here?
    self.output = nn.Linear(config.embedding_dim, config.embedding_dim)

  def forward(self, x):
    B,SL,ED = x.size()
    qkv = self.w_qkv(x)
    q, k, v = qkv.split(self.config.embedding_dim, dim=2)
    q = q.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1,2)
    k = k.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1,2)
    v = v.view(B, SL, self.config.num_heads, ED // self.config.num_heads).transpose(1,2)


    attn_out = F.scaled_dot_product_attention(q,k,v,is_causal=True)
    attn_out = attn_out.transpose(1,2).contiguous().view(B, SL, ED)
    y = self.output(attn_out)

    return y



In [10]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-04-12 14:01:10--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-04-12 14:01:10 (121 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [11]:
import tiktoken

In [12]:
with open('input.txt', 'r') as f:
      text = f.read()

enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(text)

In [13]:
class DataLoader:
  def __init__(self, B, SL, split='train'):
    self.B = B
    self.SL = SL

    with open('input.txt', 'r') as f:
      text = f.read()

    enc = tiktoken.get_encoding('gpt2')
    tokens = enc.encode(text)

    # Split the data: 90% train, 10% validation
    n = int(0.9 * len(tokens))
    if split == 'train':
        self.tokens = torch.tensor(tokens[:n])
    else:
        self.tokens = torch.tensor(tokens[n:])

    self.cursor = 0

  def get_data(self):
    buf = self.tokens[self.cursor : self.cursor + self.B*self.SL + 1]
    x = buf[:-1].view(self.B, self.SL)
    y = buf[1:].view(self.B, self.SL)
    self.cursor += self.B * self.SL

    if self.cursor + (self.B * self.SL + 1) > len(self.tokens):
      self.cursor = 0

    return x, y

In [14]:
B = 128
SL = 1024
dl_test = DataLoader(B, SL)

In [15]:
dl_test.get_data()

(tensor([[ 5962, 22307,    25,  ...,  1842,   484,  6842],
         [  514,    13,   198,  ...,   275,  1000,    13],
         [  198,    39,   603,  ...,   339,   318,   257],
         ...,
         [ 1870,   644,   531,  ...,   584,   460,    13],
         [  198,   198,    35,  ...,   750,    13,   198],
         [  198,  6369, 11357,  ...,  1549,     0,   198]]),
 tensor([[22307,    25,   198,  ...,   484,  6842,   514],
         [   13,   198,   198,  ...,  1000,    13,   198],
         [   39,   603,    11,  ...,   318,   257, 18744],
         ...,
         [  644,   531,   262,  ...,   460,    13,   198],
         [  198,    35,    52,  ...,    13,   198,   198],
         [ 6369, 11357,    25,  ...,     0,   198,   198]]))

In [16]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

In [17]:
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(5, 1)
tokens

tensor([[15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11]])

In [18]:
B = 16 # Adjust batch size based on your Colab GPU memory
SL = 256

train_loader = DataLoader(B, SL, split='train')
val_loader = DataLoader(B, SL, split='val')
input, target = train_loader.get_data()
input, target = input.to(device), target.to(device)

In [19]:
@torch.no_grad()
def estimate_loss_and_perplexity(model, loader, eval_iters=10, is_lag_model=False):
    model.eval()
    losses = torch.zeros(eval_iters)

    for k in range(eval_iters):
        X, Y = loader.get_data()
        X, Y = X.to(device), Y.to(device)

        if is_lag_model:
            logits_pre, _, _ = model(X, Y)
            V = logits_pre.size(-1)
            loss_fwd = F.cross_entropy(logits_pre.view(-1, V), Y.view(-1))
            losses[k] = loss_fwd.item()
        else:
            _, loss = model(X, Y)
            losses[k] = loss.item()

    avg_loss = losses.mean().item()
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    model.train()
    return avg_loss, perplexity

In [20]:
# # optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)
# # model.train()
# # print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
# # for step in range(500):
# #   optimizer.zero_grad()
# #   input, target = train_loader.get_data()
# #   input, target = input.to(device), target.to(device)
# #   logits_pre,logits_fut, loss = model(input, target)
# #   loss.backward()
# #   optimizer.step()
# #   print(f"step {step} loss {loss.item()}")
# #AI Generated
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# # Initial baseline
# val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader)
# print(f"Initial Baseline | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

# for step in range(1000): # Increased steps to see the curve

#   # 1. Periodically evaluate validation metrics
#   if step > 0 and step % 100 == 0:
#       val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader)
#       print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

#   # 2. Standard Training Step
#   optimizer.zero_grad()
#   input, target = train_loader.get_data()
#   input, target = input.to(device), target.to(device)

#   logits_pre, logits_fut, loss = model(input, target)
#   loss.backward()
#   optimizer.step()

#   if step % 10 == 0:
#       # Remember to divide training loss by 2 for accurate logging
#       print(f"step {step} train loss {(loss.item() / 2.0):.4f}")


In [21]:
tokens.size()

torch.Size([5, 8])

In [22]:
# #input = tokens.to(device)
# while tokens.size(1) < 100:
#   with torch.no_grad():
#     print("Output Size ",tokens.size(1))
#     logits_pre, logits_post, loss = model(input)
#     logits_pre = logits_pre[:, -1, :] #take at last position
#     probs = F.softmax(logits_pre, dim=-1) #across dimension of vocabulary
#     topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
#     random_tokens_idx = torch.multinomial(topk_probs, 1)
#     token_values = torch.gather(topk_indices, -1, random_tokens_idx)
#     tokens = torch.cat((input, token_values), dim=1)

In [23]:
# def decode_with_lag(model, prompt_tokens, max_new_tokens, temperature=1.0,
#                     confidence_threshold=0.8):

#     device = next(model.parameters()).device
#     skip_dist = model.config.lag_behind + 1

#     if not torch.is_tensor(prompt_tokens):
#         seq = torch.tensor(prompt_tokens, dtype=torch.long, device=device).unsqueeze(0)
#     else:
#         seq = prompt_tokens.to(device)
#         if seq.dim() == 1:
#             seq = seq.unsqueeze(0)

#     prompt_len = seq.size(1)
#     corrected = set()

#     for _ in range(max_new_tokens):
#         # --- Forward pass: generate next token using standard causal ---
#         logits_pre, _, _ = model(seq)
#         next_logits = logits_pre[:, -1, :] / temperature
#         probs = F.softmax(next_logits, dim=-1)
#         next_token = torch.multinomial(probs, 1)
#         seq = torch.cat([seq, next_token], dim=1)

#         # --- Backward pass: consider correcting token at current_pos - skip_dist ---
#         current_pos = seq.size(1) - 1
#         correct_pos = current_pos - skip_dist

#         if correct_pos >= prompt_len and correct_pos not in corrected:
#             # Run backward pass on the full sequence
#             _, logits_fut, _ = model(seq)

#             # Backward head at current_pos predicts t_{current_pos - skip_dist}
#             revise_logits = logits_fut[:, current_pos, :]
#             revise_probs = F.softmax(revise_logits / temperature, dim=-1)

#             proposed_token = revise_logits.argmax(dim=-1)
#             original_token = seq[:, correct_pos]

#             if proposed_token.item() != original_token.item():
#                 proposed_prob = revise_probs[0, proposed_token.item()].item()
#                 original_prob = revise_probs[0, original_token.item()].item()

#                 if proposed_prob > confidence_threshold and proposed_prob > original_prob * 1.5:
#                     seq[:, correct_pos] = proposed_token
#                     corrected.add(correct_pos)

#     return seq

In [26]:
import time

def train_and_evaluate(model, train_loader, val_loader, steps=1000, eval_interval=100, is_lag_model=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    history = {'step': [], 'train_loss': [], 'val_loss': [], 'val_ppl': []}

    # PASS FLAG HERE
    val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, is_lag_model=is_lag_model)
    print(f"Initial Baseline | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

    start_time = time.time()

    for step in range(steps):
        # PASS FLAG HERE TOO
        if step > 0 and step % eval_interval == 0:
            val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, is_lag_model=is_lag_model)
            history['step'].append(step)
            history['val_loss'].append(val_loss)
            history['val_ppl'].append(val_ppl)
            print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

        model.train()
        optimizer.zero_grad()
        X, Y = train_loader.get_data()
        X, Y = X.to(device), Y.to(device)

        if is_lag_model:
            logits_pre, logits_fut, loss = model(X, Y)
            train_loss_val = loss.item()
        else:
            logits, loss = model(X, Y)
            train_loss_val = loss.item()

        loss.backward()
        optimizer.step()

        if step % 10 == 0:
            history['train_loss'].append((step, train_loss_val))

    print(f"Training complete in {(time.time() - start_time):.2f} seconds.")
    return history

In [25]:
# 1. Train Vanilla GPT-2
print("--- TRAINING VANILLA GPT-2 ---")
# Use your vanilla GPT2 class from the Scratchpad
vanilla_model = GPT2(GPTConfig()).to(device)
vanilla_history = train_and_evaluate(
    model=vanilla_model,
    train_loader=train_loader,
    val_loader=val_loader,
    steps=2000,           # Set your desired steps here
    eval_interval=100,
    is_lag_model=False
)

# Free up GPU memory before starting the next model
del vanilla_model
torch.cuda.empty_cache()

# 2. Train Custom Lag-Behind GPT-2
# print("\n--- TRAINING LAG-BEHIND GPT-2 ---")
# # Instantiate your custom model here
# lag_model = LagBehindGPT2(GPTConfig()).to(device)
# lag_history = train_and_evaluate(
#     model=lag_model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     steps=2000,
#     eval_interval=100,
#     is_lag_model=True
# )

--- TRAINING VANILLA GPT-2 ---
Initial Baseline | Val Loss: 10.9824 | Val PPL: 58827.6641
Step  100 | Val Loss: 5.8197 | Val PPL: 336.8870
Step  200 | Val Loss: 5.3770 | Val PPL: 216.3733
Step  300 | Val Loss: 5.2389 | Val PPL: 188.4576
Step  400 | Val Loss: 5.0812 | Val PPL: 160.9696
Step  500 | Val Loss: 5.0035 | Val PPL: 148.9287
Step  600 | Val Loss: 5.0820 | Val PPL: 161.0976
Step  700 | Val Loss: 5.6847 | Val PPL: 294.3246
Step  800 | Val Loss: 5.7078 | Val PPL: 301.1982
Step  900 | Val Loss: 5.9453 | Val PPL: 381.9649


KeyboardInterrupt: 

In [27]:
del vanilla_model
torch.cuda.empty_cache()

In [28]:
# 2. Train Custom Lag-Behind GPT-2
print("\n--- TRAINING LAG-BEHIND GPT-2 ---")
# Instantiate your custom model here
lag_model = LagBehindGPT2(GPTConfig()).to(device)
lag_history = train_and_evaluate(
    model=lag_model,
    train_loader=train_loader,
    val_loader=val_loader,
    steps=900,
    eval_interval=100,
    is_lag_model=True
)


--- TRAINING LAG-BEHIND GPT-2 ---
Initial Baseline | Val Loss: 11.0378 | Val PPL: 62180.4570
Step  100 | Val Loss: 6.1509 | Val PPL: 469.1546
Step  200 | Val Loss: 5.5779 | Val PPL: 264.5049
Step  300 | Val Loss: 5.3011 | Val PPL: 200.5612
Step  400 | Val Loss: 5.0949 | Val PPL: 163.1907
Step  500 | Val Loss: 5.0489 | Val PPL: 155.8449
Step  600 | Val Loss: 4.9469 | Val PPL: 140.7415
Step  700 | Val Loss: 5.0172 | Val PPL: 150.9890
Step  800 | Val Loss: 5.1557 | Val PPL: 173.4164
Training complete in 1334.40 seconds.
